# Route measurement in a fused-readout hybrid

A standalone notebook. Colab: **Runtime → Change runtime type → GPU** first.

Six reviews of the main paper named the same gap: every result is measured
through an additive two-route readout that was designed to make route
attribution identifiable, on a 33k-parameter model. The objection writes itself
— *perhaps the phenomenon looks clean because the architecture was engineered to
expose it*.

This notebook tests the premise in an architecture that was not.

**Architecture.** Griffin-shaped: gated linear recurrence interleaved with local
attention, two layers, a residual stream and one output head. No additive route
decomposition, no per-route logits. A few million parameters rather than 33k.

**Measurement.** Because there are no per-route logits, dependence is measured
one layer earlier: resample a pathway's **output before it enters the residual
stream**. That intervention is defined for any fusion rule — it needs only that
a pathway has an identifiable output, which every residual architecture
provides.

**Task.** MQAR, the standard associative-recall benchmark from the recall-limit
literature the paper cites. It has a load knob (number of key-value pairs), and
it is the task those papers use to argue recurrent models are bottlenecked —
which makes it the hardest available test of the premise that both pathways are
individually sufficient.

## What is being asked, in order

1. **Is each pathway individually sufficient?** If recurrence alone never
   solves MQAR, attention is a requirement and route selection does not arise.
   Expect a short list; the recall-limit papers predict one.
2. **What does the free hybrid use?** Near-zero recurrent dependence with high
   attention dependence would be the main paper's §4.1 result in a fused
   readout.
3. **Does a price move it at fixed behaviour?** A rise in recurrent dependence
   at unchanged accuracy is the phenomenon. An inversion is the strong form.
4. **Do the four arms order the same way?** Only if 3 produced something to
   switch.

## Expect a null, and report it either way

This is the first experiment in the project where the measurement itself is
untested. A pre-fusion check on the small model found that fused readouts move
less under the same price than the additive one does — whether they *move* less
or are *measured* less is exactly what this notebook is for. A clean null on a
real architecture, honestly reported, is worth more than a strained positive.

Runtime: roughly 3–5 hours on a T4 at `PRESET = "full"`. Run
`PRESET = "smoke"` first — every block, tiny budgets, about ten minutes.
Results resume, so a disconnect costs at most one run.


In [ ]:
# !pip install torch numpy pandas matplotlib

import copy, hashlib, itertools, json, math, os, pathlib, random, time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================== CHOOSE HERE ==============================
PRESET  = "full"     # "smoke" (~10 min pipeline check) | "quick" | "full"
RESUME  = True
WORKDIR = os.environ.get("PAPER_WORKDIR", "/content/drive/MyDrive/hybrid")
# =========================================================================
#
# On Colab, mount Drive first so a disconnect does not cost the run:
#   from google.colab import drive; drive.mount('/content/drive')
# If you would rather not, set WORKDIR = "/content/hybrid" and download the
# results zip after each block.

RUN = {"sufficiency": True, "free": True, "price": True,
       "arms": True, "figures": True}


def S(full, quick=None, smoke=None):
    return {"full": full,
            "quick": quick if quick is not None else max(full // 3, 100),
            "smoke": smoke if smoke is not None else max(full // 30, 80)}[PRESET]


SEEDS = {"full": [0, 1, 2], "quick": [0, 1], "smoke": [0]}[PRESET]
STEPS = S(6000)          # MQAR needs more than the symbolic task
LONG  = S(16000)         # the post-convergence window
SW    = S(3000)          # where the price switches on in the arms block
LOG   = max(S(500), 50)
ACC_FLOOR = 0.90

_sfx = "" if PRESET == "full" else f"_{PRESET}"
WORK = pathlib.Path(WORKDIR).expanduser()
RESDIR, CKPTDIR = WORK / f"results{_sfx}", WORK / f"checkpoints{_sfx}"
FIGDIR = WORK / f"figures{_sfx}"
for _d in (RESDIR, CKPTDIR, FIGDIR):
    _d.mkdir(parents=True, exist_ok=True)
TAG = "hybrid"

print(f"device {DEVICE} | preset {PRESET} | seeds {SEEDS}")
print(f"work dir {WORK}")
if DEVICE.type == "cpu":
    print("!! CPU — this will take many hours. Runtime > Change runtime type > GPU.")
if PRESET == "smoke":
    print("!! SMOKE: numbers are meaningless, this only checks the pipeline runs.")


def csv(name):
    return str(RESDIR / f"{name}__{TAG}.csv")


def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)


In [ ]:
def _jsonable(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    if isinstance(o, pathlib.Path):
        return str(o)
    return str(o)


def keyof(spec):
    """Content hash of a run specification. Two runs with the same spec are the
    same experiment, so the second one is skipped."""
    return hashlib.md5(
        json.dumps(spec, sort_keys=True, default=_jsonable).encode()).hexdigest()[:16]


def cfg_tag(cfg=None):
    if cfg is None:
        return "default"
    return ",".join(f"{k}={getattr(cfg, k)}" for k in sorted(cfg.__dataclass_fields__))


class Store:
    """Append-only JSONL of finished units of work, one file per block.

    A unit is whatever a block can afford to lose: usually one (condition, seed)
    training run. Records are flushed immediately, so a killed kernel loses at
    most the run in flight. A truncated final line is discarded on load.
    """

    def __init__(self, block):
        self.block = block
        self.path = RESDIR / f"{block}__{TAG}.jsonl"
        self.recs, bad = {}, 0
        if RESUME and self.path.exists():
            for line in self.path.read_text().splitlines():
                if not line.strip():
                    continue
                try:
                    r = json.loads(line)
                    self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    bad += 1
            if bad:
                print(f"  [{block}] discarded {bad} unreadable record(s)")

    def has(self, spec):
        return RESUME and keyof(spec) in self.recs

    def get(self, spec):
        return self.recs[keyof(spec)]

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=_jsonable) + "\n")
            f.flush(); os.fsync(f.fileno())
        return r

    def frame(self):
        """Flatten to one row per result. A record either carries metric fields
        directly or a `rows` list, in which case each entry becomes a row."""
        out = []
        for r in self.recs.values():
            spec = {k: v for k, v in r["_spec"].items()}
            body = {k: v for k, v in r.items()
                    if not k.startswith("_") and k not in ("trace", "rows")}
            if "rows" in r:
                for row in r["rows"]:
                    out.append({**spec, **body, **row})
            else:
                out.append({**spec, **body})
        return pd.DataFrame(out)

    def traces(self):
        out = []
        for r in self.recs.values():
            for t in r.get("trace", []):
                out.append({**r["_spec"], **t})
        return pd.DataFrame(out)

    def elapsed(self):
        return sum(r.get("_elapsed", 0.0) for r in self.recs.values())


def cached(store, spec, fn, label=None):
    """Run `fn()` unless this exact spec is already on disk. `fn` returns the
    payload dict; anything not JSON-serialisable must be checkpointed separately."""
    if store.has(spec):
        return store.get(spec)
    t0 = time.time()
    payload = fn()
    payload["_elapsed"] = time.time() - t0
    r = store.put(spec, payload)
    if label:
        print(f"  [{store.block}] {label}  {payload['_elapsed']/60:.1f}m")
    return r


def ckpt_path(name):
    return CKPTDIR / f"{name}__{TAG}.pt"


def ckpt_save(name, obj):
    p = ckpt_path(name)
    torch.save(obj, str(p) + ".tmp")
    os.replace(str(p) + ".tmp", str(p))


def ckpt_load(name):
    p = ckpt_path(name)
    if not (RESUME and p.exists()):
        return None
    try:
        return torch.load(str(p), map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(str(p), map_location=DEVICE)
    except Exception as e:
        print(f"  checkpoint {p.name} unreadable ({e}) — recomputing")
        return None


def save_model(name, m):
    ckpt_save(name, {"state": m.state_dict()})


def load_model_into(name, m):
    ck = ckpt_load(name)
    if ck is None:
        return False
    m.load_state_dict(ck["state"])
    return True


print(f"persistence: {RESDIR}")


## Task


In [ ]:
# MQAR — multi-query associative recall, the standard benchmark for associative
# recall in hybrid architectures (Arora et al., "Zoology"). A sequence presents
# key-value pairs, then queries a subset of the keys; the model reports the
# matching values.
#
#   k1 v1 k2 v2 k3 v3 ... ? k2 ? k1
#                            ^     ^  supervised positions
#
# Two reasons this is the right task here. It has a LOAD KNOB — the number of
# pairs — which the whole framework needs, and it is the task the recall-limit
# literature uses to show that recurrent models are bottlenecked. That makes it
# the hardest available test of the premise that both routes are individually
# sufficient: if the recurrent alternative is viable anywhere on MQAR, the
# premise holds in the regime those papers say it should not.

@dataclass
class Config:
    vocab_kv: int = 64          # symbols usable as keys or values
    d_model: int = 128
    n_layers: int = 2
    n_heads: int = 4
    window: int = 32            # local attention span
    lr: float = 1e-3
    batch_size: int = 64
    grad_clip: float = 1.0


CFG = Config()
QUERY, PAD = 0, 1
N_SPECIAL = 2
VOCAB = N_SPECIAL + CFG.vocab_kv
IGNORE = -100


class MQAR:
    """`n_pairs` key-value pairs, then `n_query` queries drawn from those keys.

    Keys and values are disjoint halves of the alphabet so a model cannot
    shortcut by copying the most recent token. Only query-answer positions are
    supervised; everything else is masked out of the loss.
    """
    name = "mqar"

    def __init__(self, cfg=None, n_query=4):
        self.cfg = cfg or CFG
        self.n_query = n_query
        h = self.cfg.vocab_kv // 2
        self.keys = torch.arange(N_SPECIAL, N_SPECIAL + h)
        self.vals = torch.arange(N_SPECIAL + h, N_SPECIAL + 2 * h)

    @property
    def n_out(self):
        return VOCAB

    def generate(self, B, n_pairs, device=DEVICE):
        nq = min(self.n_query, n_pairs)
        ki = torch.stack([torch.randperm(len(self.keys))[:n_pairs] for _ in range(B)])
        vi = torch.randint(0, len(self.vals), (B, n_pairs))
        k, v = self.keys[ki].to(device), self.vals[vi].to(device)

        body = torch.stack([k, v], -1).reshape(B, 2 * n_pairs)     # k1 v1 k2 v2 ...
        qsel = torch.stack([torch.randperm(n_pairs)[:nq] for _ in range(B)]).to(device)
        ar = torch.arange(B, device=device).unsqueeze(1)
        qk, qv = k[ar, qsel], v[ar, qsel]
        qtok = torch.full((B, nq), QUERY, device=device, dtype=torch.long)
        tail = torch.stack([qtok, qk], -1).reshape(B, 2 * nq)      # ? k ? k ...

        x = torch.cat([body, tail], 1)
        # the answer to "? k" is supervised at the position holding k
        y = torch.full_like(x, IGNORE)
        pos = 2 * n_pairs + 1 + 2 * torch.arange(nq, device=device)
        y[ar, pos.unsqueeze(0).expand(B, -1)] = qv
        return {"x": x, "y": y, "answer_pos": pos}


def make_task(n_query=4):
    return MQAR(n_query=n_query)


_t = make_task()
_b = _t.generate(2, 4)
print(f"MQAR: vocab {VOCAB}, {len(_t.keys)} keys, {len(_t.vals)} values")
print(f"  example (4 pairs, {_t.n_query} queries), tokens and targets:")
for i in range(2):
    xs = " ".join(f"{int(t):>3}" for t in _b["x"][i])
    ys = " ".join("  ." if int(t) == IGNORE else f"{int(t):>3}" for t in _b["y"][i])
    print(f"    x {xs}")
    print(f"    y {ys}")
print(f"  chance accuracy on answers: {1/len(_t.vals):.3f}")


## The architecture, and where the measurement goes

The key design decision is that this model has **no additive route
decomposition**. Routes are fused by the residual stream, exactly as they are in
the architectures the paper's introduction names. So the ablation cannot target
a route's logit contribution — that quantity does not exist here.

Instead it targets the pathway's write into the stream. `HybridBlock.forward`
computes `x + ablate(rec_output)` and `x + ablate(att_output)`, so resampling
either one replaces that pathway's contribution for this example with another
example's, leaving everything else intact. This is the same resample-ablation
the paper uses, applied one layer earlier.


In [ ]:
# A Griffin-shaped hybrid: gated linear recurrence interleaved with local
# attention, two layers, standard residual stream and a single output head.
#
# This is deliberately NOT the additive two-route readout the rest of the paper
# uses. The point of this experiment is to test whether route usage is
# measurable in an architecture people deploy, where the routes are fused by a
# residual stream rather than summed at the logits. The measurement therefore
# has to happen one layer earlier: resample a block's OUTPUT before it is added
# to the stream. That intervention is defined for any fusion rule.

def ablate(v, spec):
    """Resample across the batch: a sequence's activation is replaced with
    another sequence's, so the stream still receives an in-distribution vector
    that carries no information about this example. Zeroing would be
    off-manifold, which matters more here than in a small model."""
    if v is None or spec is None:
        return v
    mode, strength = spec if isinstance(spec, (tuple, list)) else (spec, 1.0)
    if strength <= 0:
        return v
    repl = {"shuffle": lambda: v[torch.randperm(v.shape[0], device=v.device)],
            "zero": lambda: torch.zeros_like(v)}[mode]()
    return (1.0 - strength) * v + strength * repl


class GatedRecurrence(nn.Module):
    """Real-gated linear recurrence, the Griffin/Hawk-style recurrent block:
    h_t = a_t * h_{t-1} + (1 - a_t) * (i_t * x_t), with an output gate. The
    scan is sequential — fine at these lengths, and it keeps the block readable."""

    def __init__(self, d, expand=1.5):
        super().__init__()
        di = int(d * expand)
        self.in_proj = nn.Linear(d, 2 * di)
        self.a_proj = nn.Linear(d, di)
        self.i_proj = nn.Linear(d, di)
        self.conv = nn.Conv1d(di, di, 4, groups=di, padding=3)
        self.out_proj = nn.Linear(di, d)
        self.di = di

    def forward(self, x):
        B, T, _ = x.shape
        u, gate = self.in_proj(x).chunk(2, dim=-1)
        u = self.conv(u.transpose(1, 2))[:, :, :T].transpose(1, 2)
        a = torch.sigmoid(self.a_proj(x))          # forget gate, in (0, 1)
        i = torch.sigmoid(self.i_proj(x))          # input gate
        h = x.new_zeros(B, self.di)
        out = []
        for t in range(T):
            h = a[:, t] * h + (1 - a[:, t]) * (i[:, t] * u[:, t])
            out.append(h)
        y = torch.stack(out, 1) * F.silu(gate)
        return self.out_proj(y)


class LocalAttention(nn.Module):
    """Causal attention over a sliding window, as in Griffin's local-attention
    blocks. The window is what makes it local rather than global; at these
    sequence lengths it still spans the whole context for small loads, which is
    the regime where retrieval should be easiest."""

    def __init__(self, d, n_heads=4, window=32):
        super().__init__()
        self.h, self.dh, self.w = n_heads, d // n_heads, window
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = (t.view(B, T, self.h, self.dh).transpose(1, 2) for t in (q, k, v))
        sc = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        idx = torch.arange(T, device=x.device)
        rel = idx[:, None] - idx[None, :]
        mask = (rel < 0) | (rel >= self.w)          # causal, windowed
        sc = sc.masked_fill(mask[None, None], float("-inf"))
        y = (sc.softmax(-1) @ v).transpose(1, 2).reshape(B, T, D)
        return self.proj(y)


class HybridBlock(nn.Module):
    """One layer: recurrence, then local attention, each with its own norm and
    residual. `intervene` resamples a block's contribution before it enters the
    stream — the fused-readout analogue of ablating a route's logit."""

    def __init__(self, cfg):
        super().__init__()
        d = cfg.d_model
        self.n1, self.n2, self.n3 = (nn.LayerNorm(d) for _ in range(3))
        self.rec = GatedRecurrence(d)
        self.att = LocalAttention(d, cfg.n_heads, cfg.window)
        self.ffn = nn.Sequential(nn.Linear(d, 3 * d), nn.GELU(), nn.Linear(3 * d, d))

    def forward(self, x, intervene=None, collect=None):
        iv = intervene or {}
        r = self.rec(self.n1(x))
        if collect is not None:
            collect["rec"].append(r)
        x = x + ablate(r, iv.get("recurrent"))
        a = self.att(self.n2(x))
        if collect is not None:
            collect["att"].append(a)
        x = x + ablate(a, iv.get("attention"))
        return x + self.ffn(self.n3(x))


class HybridLM(nn.Module):
    def __init__(self, cfg=None, n_out=None):
        super().__init__()
        cfg = cfg or CFG
        self.cfg = cfg
        self.emb = nn.Embedding(VOCAB, cfg.d_model)
        self.pos = nn.Parameter(torch.zeros(1, 512, cfg.d_model))
        nn.init.normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([HybridBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, n_out or VOCAB)

    def forward(self, x, intervene=None, collect_parts=False):
        h = self.emb(x) + self.pos[:, :x.shape[1]]
        parts = {"rec": [], "att": []} if collect_parts else None
        for b in self.blocks:
            h = b(h, intervene, parts)
        out = {"logits": self.head(self.norm(h))}
        if collect_parts:
            # the magnitude each pathway writes into the residual stream: the
            # fused-readout analogue of the additive readout's contribution norm
            out["rec_norm"] = float(torch.stack([r.norm(dim=-1).mean() for r in parts["rec"]]).mean())
            out["att_norm"] = float(torch.stack([a.norm(dim=-1).mean() for a in parts["att"]]).mean())
        return out


def route_params(m):
    rec = [p for b in m.blocks for p in b.rec.parameters()]
    att = [p for b in m.blocks for p in b.att.parameters()]
    seen = {id(p) for p in rec + att}
    shared = [p for p in m.parameters() if id(p) not in seen]
    return rec, att, shared


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


def build(variant="hybrid", cfg=None, n_out=None):
    """`hybrid` has both pathways; `recurrent_only` and `attention_only` delete
    one, to test whether each is individually sufficient."""
    cfg = cfg or CFG
    m = HybridLM(cfg, n_out)
    if variant == "recurrent_only":
        for b in m.blocks:
            b.att = None
            b.forward = (lambda x, intervene=None, collect=None, b=b:
                         (lambda y: y + b.ffn(b.n3(y)))(
                             x + ablate(b.rec(b.n1(x)), (intervene or {}).get("recurrent"))))
    elif variant == "attention_only":
        for b in m.blocks:
            b.rec = None
            b.forward = (lambda x, intervene=None, collect=None, b=b:
                         (lambda y: y + b.ffn(b.n3(y)))(
                             x + ablate(b.att(b.n2(x)), (intervene or {}).get("attention"))))
    return m


for v in ["hybrid", "recurrent_only", "attention_only"]:
    print(f"{v:<16} {count_params(build(v)):>10,} parameters")


## Training and measurement

The price is on the mean norm of what the attention blocks write into the
residual stream. In the additive-readout model the analogous quantity is the
retrieval route's logit contribution; here there is no per-route logit, so the
pathway's contribution to the stream is the closest available analogue — and it
is what a fused architecture actually exposes. Say so in the paper; the two
prices are not identical and the difference matters for comparing coefficients.

`att_share` is the cheap proxy — the fraction of stream-write magnitude coming
from attention — reported alongside the causal dependence so the same
proxy-versus-necessity comparison the paper makes elsewhere is available here.


In [ ]:
def train(n_pairs, seed, steps=None, lam=0.0, variant="hybrid", lam_after=None,
          switch=None, log_every=None, ckpt_key=None, ckpt_every=2000, n_query=4):
    """The price is on the mean norm of what the attention blocks write into the
    residual stream. In the additive-readout model the equivalent quantity is
    the retrieval route's logit contribution; here there is no per-route logit,
    so the pathway's contribution to the stream is the closest analogue and it
    is what a fused architecture actually exposes."""
    steps = steps or STEPS
    task = make_task(n_query)
    set_seed(seed)
    m = build(variant, n_out=task.n_out).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr, weight_decay=0.0)
    trace, start = [], 1

    if ckpt_key is not None:
        ck = ckpt_load(f"run_{ckpt_key}")
        if ck is not None and ck.get("steps") == steps:
            m.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
            trace, start = ck["trace"], ck["step"] + 1

    for step in range(start, steps + 1):
        cur = lam if (switch is None or step <= switch) else lam_after
        m.train()
        b = task.generate(CFG.batch_size, n_pairs)
        need = cur and cur > 0 and variant == "hybrid"
        o = m(b["x"], collect_parts=need)
        loss = F.cross_entropy(o["logits"].reshape(-1, o["logits"].shape[-1]),
                               b["y"].reshape(-1), ignore_index=IGNORE)
        if need:
            parts = {"rec": [], "att": []}
            h = m.emb(b["x"]) + m.pos[:, :b["x"].shape[1]]
            for blk in m.blocks:
                h = blk(h, None, parts)
            pen = torch.stack([a.norm(dim=-1).mean() for a in parts["att"]]).mean()
            loss = loss + cur * pen
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip); opt.step()

        if log_every and (step % log_every == 0 or step == 1):
            trace.append({"step": step, **measure(m, task, n_pairs, n_batches=3)})
        if ckpt_key is not None and step % ckpt_every == 0 and step < steps:
            ckpt_save(f"run_{ckpt_key}", {"model": m.state_dict(), "opt": opt.state_dict(),
                                          "step": step, "steps": steps, "trace": trace})
    if ckpt_key is not None and ckpt_path(f"run_{ckpt_key}").exists():
        ckpt_path(f"run_{ckpt_key}").unlink()
    return m, task, pd.DataFrame(trace)


@torch.no_grad()
def accuracy(m, task, n_pairs, intervene=None, n_batches=8):
    """Accuracy on the supervised answer positions only."""
    m.eval(); c = t = 0
    for _ in range(n_batches):
        b = task.generate(CFG.batch_size, n_pairs)
        pred = m(b["x"], intervene=intervene)["logits"].argmax(-1)
        sel = b["y"] != IGNORE
        c += (pred[sel] == b["y"][sel]).sum().item(); t += int(sel.sum())
    m.train(); return c / max(t, 1)


@torch.no_grad()
def measure(m, task, n_pairs, n_batches=8, mode="shuffle"):
    """Route dependence through a FUSED readout. The intervention resamples a
    pathway's output before it is added to the residual stream, so it needs no
    additive decomposition of the logits — only that the pathway has an
    identifiable output, which every residual architecture provides."""
    chance = 1.0 / len(task.vals)
    acc = accuracy(m, task, n_pairs, n_batches=n_batches)
    den = max(acc - chance, 1e-6)
    d = lambda a: float(np.clip((acc - a) / den, 0, 1))
    has_rec = m.blocks[0].rec is not None
    has_att = m.blocks[0].att is not None
    out = {"acc_clean": acc,
           "rec_dep": d(accuracy(m, task, n_pairs, {"recurrent": (mode, 1.0)},
                                 n_batches)) if has_rec else 0.0,
           "att_dep": d(accuracy(m, task, n_pairs, {"attention": (mode, 1.0)},
                                 n_batches)) if has_att else 0.0}
    if has_rec and has_att:
        m.eval(); rn, an = [], []
        for _ in range(3):
            o = m(task.generate(CFG.batch_size, n_pairs)["x"], collect_parts=True)
            rn.append(o["rec_norm"]); an.append(o["att_norm"])
        m.train()
        r, a = float(np.mean(rn)), float(np.mean(an))
        out["rec_norm"], out["att_norm"] = r, a
        # the cheap proxy, for the same comparison the paper makes elsewhere
        out["att_share"] = a / (r + a + 1e-9)
    else:
        out["rec_norm"] = out["att_norm"] = out["att_share"] = np.nan
    return out


def boot_ci(vals, n=10000, alpha=0.05, seed=0):
    v = np.asarray([x for x in vals if x == x], float)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    mu = rng.choice(v, size=(n, len(v)), replace=True).mean(1)
    return float(v.mean()), float(np.quantile(mu, alpha/2)), float(np.quantile(mu, 1-alpha/2))


## The experiments


In [ ]:
PAIRS = [4, 8, 16, 32] if PRESET == "full" else [4, 8]

# ---- 1. is each pathway individually sufficient? --------------------------
# The premise the whole framework rests on. MQAR is the task the recall-limit
# literature uses to argue that recurrent models are bottlenecked, so this is
# the hardest available test of it: wherever recurrence alone solves the task,
# the hybrid's choice of attention is a choice among sufficient algorithms.
if RUN["sufficiency"]:
    st = Store("sufficiency")
    for v, n, sd in itertools.product(["recurrent_only", "attention_only"],
                                      PAIRS, SEEDS):
        spec = {"block": "sufficiency", "variant": v, "n_pairs": n, "seed": sd,
                "steps": STEPS}
        cached(st, spec,
               lambda v=v, n=n, sd=sd: (lambda mt: measure(mt[0], mt[1], n))(
                   train(n, sd, variant=v)[:2]),
               label=f"{v:<15} {n:>2} pairs seed {sd}")
    SUF = st.frame(); SUF.to_csv(csv("h1_sufficiency"), index=False)
    print("\naccuracy of each pathway alone:")
    print(SUF.pivot_table(index="variant", columns="n_pairs",
                          values="acc_clean").round(3).to_string())
    rec = SUF[SUF.variant == "recurrent_only"].groupby("n_pairs").acc_clean.mean()
    VIABLE = [int(n) for n, a in rec.items() if a >= 0.95]
    print(f"\n  loads where recurrence alone reaches 0.95: {VIABLE or 'none'}")
    print("  The recall-limit literature predicts this list is short. Where it is")
    print("  non-empty, both algorithms are available and the hybrid's preference")
    print("  is a choice; where it is empty, attention is a requirement and the")
    print("  question of route selection does not arise.")
else:
    VIABLE = PAIRS[:1]

# ---- 2. what does the free hybrid use? ------------------------------------
if RUN["free"]:
    st = Store("free")
    for n, sd in itertools.product(PAIRS, SEEDS):
        spec = {"block": "free", "n_pairs": n, "seed": sd, "steps": STEPS}
        cached(st, spec,
               lambda n=n, sd=sd: (lambda mt: measure(mt[0], mt[1], n))(
                   train(n, sd)[:2]),
               label=f"free {n:>2} pairs seed {sd}")
    FR = st.frame(); FR.to_csv(csv("h2_free"), index=False)
    print("\nthe free hybrid, no price:")
    print(FR.groupby("n_pairs")[["acc_clean", "rec_dep", "att_dep",
                                 "att_share"]].agg(["mean", "std"]).round(3).to_string())
    print("\n  rec_dep near zero with att_dep high is the §4.1 result in a fused")
    print("  architecture: with both pathways available and neither priced, the")
    print("  model routes the computation through attention.")

# ---- 3. does a price move it, at fixed behaviour? -------------------------
if RUN["price"]:
    LAMS = [0.0, 0.003, 0.01, 0.03, 0.1]
    st = Store("price")
    for n, lam, sd in itertools.product(VIABLE[:2] or PAIRS[:1], LAMS, SEEDS):
        spec = {"block": "price", "n_pairs": n, "lam": lam, "seed": sd,
                "steps": LONG}
        cached(st, spec,
               lambda n=n, lam=lam, sd=sd, spec=spec: (
                   lambda mt: measure(mt[0], mt[1], n))(
                       train(n, sd, steps=LONG, lam=lam,
                             ckpt_key=keyof(spec))[:2]),
               label=f"price lam={lam:<6} {n:>2} pairs seed {sd}")
    PR = st.frame(); PR.to_csv(csv("h3_price"), index=False)
    print("\nunder price:")
    print(PR.pivot_table(index="lam", columns="n_pairs",
                         values=["acc_clean", "rec_dep", "att_dep", "att_share"])
          .round(3).to_string())
    for n in sorted(PR.n_pairs.unique()):
        g = PR[PR.n_pairs == n]
        hi = g[g.lam == g.lam.max()]
        lo = g[g.lam == 0.0]
        n_inv = int((hi.rec_dep.values > hi.att_dep.values).sum())
        print(f"\n  {n} pairs: rec_dep {lo.rec_dep.mean():.3f} -> {hi.rec_dep.mean():.3f}"
              f"   att_dep {lo.att_dep.mean():.3f} -> {hi.att_dep.mean():.3f}"
              f"   acc {hi.acc_clean.mean():.3f}   {n_inv}/{len(hi)} seeds inverted")
    print("\n  A rise in rec_dep at unchanged accuracy is the phenomenon in a")
    print("  fused readout. An inversion is the strong form; a rise without one")
    print("  is still evidence, and should be reported as the weaker claim.")

# ---- 4. the four arms, if there is anything to switch ---------------------
if RUN["arms"] and RUN["price"]:
    best = None
    for n in sorted(PR.n_pairs.unique()):
        g = PR[(PR.n_pairs == n) & (PR.lam == PR.lam.max())]
        if len(g) and g.acc_clean.mean() >= ACC_FLOOR:
            gain = g.rec_dep.mean() - PR[(PR.n_pairs == n) & (PR.lam == 0.0)].rec_dep.mean()
            if best is None or gain > best[1]:
                best = (n, gain, PR.lam.max())
    if best is None:
        print("\nno condition moved the route at acceptable accuracy; the four-arm")
        print("test needs one, so it is skipped. Report the price sweep alone.")
    else:
        n_arm, _, lam_arm = best
        print(f"\nfour arms at {n_arm} pairs, lambda {lam_arm} "
              f"(the condition with the largest route change at ceiling)")
        ARMS = {"maintained": (lam_arm, lam_arm), "withdrawn": (lam_arm, 0.0),
                "never": (0.0, 0.0), "late": (0.0, lam_arm)}
        st = Store("arms")
        for (arm, (lb, la)), sd in itertools.product(ARMS.items(), SEEDS):
            spec = {"block": "arms", "arm": arm, "n_pairs": n_arm, "lam": lam_arm,
                    "seed": sd, "switch": SW, "steps": LONG}

            def _go(arm=arm, lb=lb, la=la, sd=sd, spec=spec):
                m, task, tr = train(n_arm, sd, steps=LONG, lam=lb, lam_after=la,
                                    switch=SW, log_every=LOG, ckpt_key=keyof(spec))
                at = tr[tr.step <= SW].att_share.iloc[-1] if len(tr) else np.nan
                return {"share_at_switch": float(at), **measure(m, task, n_arm),
                        "trace": tr.to_dict("records")}

            cached(st, spec, _go, label=f"{arm:<11} seed {sd}")
        AR = st.frame(); AR.to_csv(csv("h4_arms"), index=False)
        st.traces().to_csv(csv("h4_arms_traces"), index=False)
        print()
        for arm in ["never", "maintained", "late", "withdrawn"]:
            g = AR[AR.arm == arm]
            if len(g):
                print(f"  {arm:<11} att_share {g.share_at_switch.mean():.3f} -> "
                      f"{g.att_share.mean():.3f}+/-{g.att_share.std():.3f}"
                      f"   rec_dep {g.rec_dep.mean():.3f}"
                      f"   acc {g.acc_clean.mean():.4f}   n={len(g)}")
        print("\n  never  -> no drift without a price")
        print("  late   -> the price applied after convergence still moves the route")
        print("  withdrawn -> the change reverses, so the price is required")

import shutil
shutil.make_archive(str(WORK / "results_backup"), "zip", RESDIR)
print(f"\nbacked up to {WORK}/results_backup.zip")
